In [1]:
import pandas as pd

df = pd.read_csv('../data/temperature.csv')
df.head()

,datetime,Vancouver,Portland,San Francisco,Seattle,Los Angeles,San Diego,Las Vegas,Phoenix,Albuquerque,...,Philadelphia,New York,Montreal,Boston,Beersheba,Tel Aviv District,Eilat,Haifa,Nahariyya,Jerusalem
0,2012-10-01 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,309.100000,NaN,NaN,NaN
1,2012-10-01 13:00:00,284.630000,282.080000,289.480000,281.800000,291.870000,291.530000,293.410000,296.600000,285.120000,...,285.630000,288.220000,285.830000,287.170000,307.590000,305.470000,310.580000,304.4,304.4,303.5
2,2012-10-01 14:00:00,284.629041,282.083252,289.474993,281.797217,291.868186,291.533501,293.403141,296.608509,285.154558,...,285.663208,288.247676,285.834650,287.186092,307.590000,304.310000,310.495769,304.4,304.4,303.5
3,2012-10-01 15:00:00,284.626998,282.091866,289.460618,281.789833,291.862844,291.543355,293.392177,296.631487,285.233952,...,285.756824,288.326940,285.847790,287.231672,307.391513,304.281841,310.411538,304.4,304.4,303.5
4,2012-10-01 16:00:00,284.624955,282.100481,289.446243,281.782449,291.857503,291.553209,293.381213,296.654466,285.313345,...,285.850440,288.406203,285.860929,287.277251,307.145200,304.238015,310.327308,304.4,304.4,303.5


In [2]:
print(df.shape)


(45253, 37)


In [3]:
print(df.isnull().sum())

datetime               0
Vancouver            795
Portland               1
San Francisco        793
Seattle                3
Los Angeles            3
San Diego              1
Las Vegas              1
Phoenix                3
Albuquerque            1
Denver                 1
San Antonio            1
Dallas                 4
Houston                3
Kansas City            1
Minneapolis           13
Saint Louis            1
Chicago                3
Nashville              2
Indianapolis           7
Atlanta                6
Detroit                1
Jacksonville           1
Charlotte              3
Miami                805
Pittsburgh             3
Toronto                1
Philadelphia           3
New York             793
Montreal               3
Boston                 3
Beersheba            798
Tel Aviv District    793
Eilat                792
Haifa                798
Nahariyya            797
Jerusalem            793
dtype: int64


In [4]:
df_ny = df[['datetime', 'New York']].copy()
df_ny.columns = ['datetime', 'temperature']
df_ny.head()

,datetime,temperature
0,2012-10-01 12:00:00,NaN
1,2012-10-01 13:00:00,288.220000
2,2012-10-01 14:00:00,288.247676
3,2012-10-01 15:00:00,288.326940
4,2012-10-01 16:00:00,288.406203


In [5]:
# Convert Kelvin to Celsius
df_ny['temperature'] = df_ny['temperature'] - 273.15

# Fill missing values with the average temperature
df_ny['temperature'] = df_ny['temperature'].fillna(df_ny['temperature'].mean())

# Confirm no more missing values
print(df_ny.isnull().sum())

datetime       0
temperature    0
dtype: int64


In [6]:
df_ny.head(10)


,datetime,temperature
0,2012-10-01 12:00:00,12.250406
1,2012-10-01 13:00:00,15.070000
2,2012-10-01 14:00:00,15.097676
3,2012-10-01 15:00:00,15.176940
4,2012-10-01 16:00:00,15.256203
5,2012-10-01 17:00:00,15.335467
6,2012-10-01 18:00:00,15.414730
7,2012-10-01 19:00:00,15.493994
8,2012-10-01 20:00:00,15.573257
9,2012-10-01 21:00:00,15.652521


In [7]:
df_ny['datetime'] = pd.to_datetime(df_ny['datetime'])

df_ny['hour'] = df_ny['datetime'].dt.hour
df_ny['month'] = df_ny['datetime'].dt.month
df_ny['day_of_week'] = df_ny['datetime'].dt.dayofweek
df_ny['temp_lag1'] = df_ny['temperature'].shift(1)

df_ny = df_ny.dropna()
df_ny.head()

,datetime,temperature,hour,month,day_of_week,temp_lag1
1,2012-10-01 13:00:00,15.070000,13,10,0,12.250406
2,2012-10-01 14:00:00,15.097676,14,10,0,15.070000
3,2012-10-01 15:00:00,15.176940,15,10,0,15.097676
4,2012-10-01 16:00:00,15.256203,16,10,0,15.176940
5,2012-10-01 17:00:00,15.335467,17,10,0,15.256203


In [8]:
X = df_ny[['hour', 'month', 'day_of_week', 'temp_lag1']]
y = df_ny['temperature']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (45252, 4)
Target shape: (45252,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (36201, 4)
Testing size: (9051, 4)


In [10]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [11]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:.2f}°C")
print(f"R2 Score: {r2:.2f}")

Mean Absolute Error: 0.68°C
R2 Score: 0.99
